# Entrenar Modelos
En este programa se entrenarán los modelos K-NN, SVM, Naive Bayes y Random Forest. Estos tendrán un entrenamiento por cada dataset diferente generado por la práctica 2, usando validación cruzada de 5 pliegues. El resultado será devuelto en el siguiente sistema de carpetas. Primero, una carpeta por modelo, dentro de cada una tendremos una carpeta por cada dataset diferente generado, y dentro de cada una de estas tendremos un archivo de modelo por cada pliegue de kfold.

## Cargamos Librerias
Cargamos las librerias necesarias para el proyecto

In [62]:
import pandas as pd
import sklearn as sk
import joblib
import pathlib as pl
import os

## Función Entrenar modelo con kfold
Vamos a definir una función llamada train_model_csv() la cual acepte los siguientes parámetros:
- model: Modelo a entrenar por la función
- data_csv: Archivo csv a usar para el entrenamiento del modelo
- output_file: Ruta del archivo a devolver por el entrenamiento

In [63]:
def train_model_csv(model, data_csv, output_file):
	with open(data_csv, 'r') as f:
		data = pd.read_csv(f)
	X = data.drop('species', axis=1)
	Y = data['species']
	model.fit(X, Y)
	joblib.dump(model, output_file)

Ahora vamos a definir una funcion que dado un modelo entrene todas las variantes de datasets y sus kfolds usando train_model_csv

- Nombre: train_model
- Parámetros:
    - model: Modelo limpio a usar para entrenar
    - model_name: Nombre del modelo

In [ ]:
def train_model(model, model_name):
    tipos = ['original', 'estandarizado', 'normalizado']
    variantes = ['', '_PCA95', '_PCA80']
    for tipo in tipos:
        for var in variantes:
            ruta = pl.Path(f'./kfolds_data/conj_{tipo}{var}/')
            training_csv = sorted([x.name for x in ruta.glob('training*.csv')])
            for train_file in training_csv:
                data_csv = ruta / train_file
                os.makedirs(f'./trained_models/{model_name}/{tipo}{var}', exist_ok=True)
                train_file_no_extension = train_file.split('.')[0]
                output_file = f'./trained_models/{model_name}/{tipo}{var}/{model_name}_{train_file_no_extension}.joblib'
                fresh_model = sk.clone(model)
                train_model_csv(fresh_model, data_csv, output_file)

## Entrenamos todos los modelos
Por cada modelo correspondiente, iteramos por cada uno de los .csv y llamamos a la funcion train_model con los parametros correspondientes.

### Entrenamos K-NN

In [65]:
knn_model = sk.neighbors.KNeighborsClassifier(n_neighbors=5)
train_model(knn_model, 'KNN')

### Entrenamos SVM

In [66]:
svm_model = sk.svm.SVC(kernel='rbf', C=1.0, gamma='scale')
train_model(svm_model, 'SVM')

### Entrenamos Naive Bayes

In [67]:
naive_bayes_model = sk.naive_bayes.GaussianNB()
train_model(naive_bayes_model, 'NaiveBayes')

### Entrenamos Random Forest

In [68]:
random_forest_model = sk.ensemble.RandomForestClassifier(n_estimators=100, random_state=42)
train_model(random_forest_model, 'RandomForest')